In [6]:
import os
from typing import BinaryIO
from tqdm import tqdm


def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))


def init_worker(num_workers=8):
    pid = os.getpid()
    # Restrict to a subset of CPUs, e.g., cores 0-3
    os.sched_setaffinity(pid, set(list(range(num_workers))))

In [43]:
data_dir = "/scratch/shared/beegfs/piyush/datasets/text_data"

mode = "train"
filepath = f"{data_dir}/TinyStoriesV2-GPT4-{mode}.txt"
print("Loading: ", filepath)


import regex as re
# from collections import defaultdict
from collections import Counter


def pretokenize_document(d: str):
    """
    Uses GPT2 regex-based pre-tokenization and returns a dict with pre token mapped to its count.
    """
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    match_iter = re.finditer(re.compile(PAT), d)
    counts = Counter()
    for m in match_iter:
        counts[d[m.start():m.end()]] += 1
    return counts


def pretokenize_chunk(chunk: str, special_tokens=['<|endoftext|>']):
    # Split documents around special tokens
    pattern = "|".join(re.escape(token) for token in special_tokens)
    pattern = re.compile(pattern)
    docs = re.split(pattern, chunk)

    # Run pre-tokenization on each document and gather a single count dict
    counts_chunk = Counter()
    iterator = tqdm(docs, desc="Running pre-tokenization for documents in a chunk")
    for d in iterator:
        counts_chunk += pretokenize_document(d)
    return counts_chunk


# with open(filepath, "rb") as f:
#     num_processes = 8
#     boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

#     # The following is a serial implementation, but you can parallelize this
#     # by sending each start/end pair to a set of processes.
#     counts = defaultdict(int)
#     for start, end in zip(boundaries[:-1], boundaries[1:]):
#         f.seek(start)
#         chunk = f.read(end - start).decode("utf-8", errors="ignore")
#         counts.update(pretokenize_chunk(chunk))

# len(counts)


# Parallelly process multiple chunks

from multiprocessing import Pool

def process_chunk(args):
    filepath, start, end = args
    with open(filepath, "rb") as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
    return pretokenize_chunk(chunk)

with open(filepath, "rb") as f:
    num_processes = 8
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

# Each worker opens the file independently and reads only its chunk
tasks = [(filepath, start, end) for start, end in zip(boundaries[:-1], boundaries[1:])]

with Pool(num_processes, initializer=init_worker) as pool:
    chunk_counts = pool.map(process_chunk, tasks)

counts = defaultdict(int)
for c in chunk_counts:
    for k, v in c.items():
        counts[k] += v

Loading:  /scratch/shared/beegfs/piyush/datasets/text_data/TinyStoriesV2-GPT4-train.txt


Running pre-tokenization for documents in a chunk: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 340083/340083 [05:56<00:00, 953.12it/s]
Running pre-tokenization for documents in a chunk: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339632/339632 [05:58<00:00, 948.42it/s]
Running pre-tokenization for documents in a chunk: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339463/339463 [05:58<00:00, 945.80it/s]
Running pre-tokenization for documents in a chunk: 100%|███████████████████████

In [44]:
for x in chunk_counts:
    print(len(x))

30009
29993
29854
30050
30004
30141
30065
29916


In [45]:
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
sorted_counts[:20]

[('.', 41764510),
 (',', 23284330),
 (' the', 20828576),
 (' and', 19475966),
 (' a', 15063529),
 ('\n', 15011026),
 (' to', 14903559),
 (' was', 10593232),
 (' They', 5226508),
 (' it', 5138607),
 (' He', 4840028),
 (' "', 4774529),
 (' The', 4623365),
 (' said', 4370380),
 (' day', 4216755),
 (' with', 4208768),
 (' in', 3848306),
 (' She', 3843343),
 (' her', 3840758),
 (' his', 3781381)]

In [47]:
len(counts)

59933

In [48]:
import torch

save_name = f"TinyStoriesV2-GPT4-{mode}-pretokenization_counts.pt"
save_path = f"{data_dir}/{save_name}"
torch.save(counts, save_path)

In [46]:
# Verify
query = ' day'
with open(filepath, "rb") as f:
    num_processes = 8
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

    # The following is a serial implementation, but you can parallelize this
    # by sending each start/end pair to a set of processes.
    c = 0
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        match_iter = re.finditer(re.compile(query), chunk)
        for m in match_iter:
            c += 1
        # del chunk
    print(f"Estimated number of `{query}`: ", c)
    print(f"Number of `{query}` from pretok: ", counts[query])

Estimated number of ` day`:  4268218
Number of ` day` from pretok:  4216755
